# Processing a folder of files with a Python script

You have a folder of files you would like to process with a Python script. This recipe will take you through the process of placing a workload of multiple jobs at an HTCondor Access Point where each job processes one file. The recipe assumes that the script takes the path name of the file to be processed as a command line argument. 

The workload components (container, data folder, Python script) have already been uploaded and placed as follows: 

* Python scripts (.py) in the `scripts` folder. 
* Data files (.csv) in the `inputs` folder. 

1. Create directories for output files produced by the script. 

In [ ]:
mkdir results

2. Create a workload description using the HTCondor Workload Description Language (WDL) to process a subset of the files.  

In [ ]:
# Create the job list (input file ids)
./generate_table.sh

Click to view the job table:

<button data-commandLinker-command="docmanager:open"
        data-commandLinker-args='{"path": "recipe-demos/jobtable.csv"}' href="#">
  Open jobtable.csv
</button>

In [ ]:
# Create a job template that reads in the job list
cat << 'EOF' > jobs.sub
should_transfer_files  = YES
transfer_input_files   = scripts/, inputs/$(INFILE)
transfer_output_files  = $(OUTFILE)
transfer_output_remaps = "$(OUTFILE)=results/$(OUTFILE)"

log             = logs/$(CLUSTER).log
error           = logs/$(CLUSTER).$(PROCID).err
output          = logs/$(CLUSTER).$(PROCID).out

EOF

3. Place the test workload by feeding in the job table. We will only run the first 3 lines as a test. 

In [ ]:
condor_submit jobs.sub -table '[0:3] jobtable.csv'

In [ ]:
# check if jobs are running
condor_q

4. Once completed, review the results. 

In [ ]:
ls -lh results/

In [ ]:
# are all the output files created? 
echo "Number of outputs:" `ls results | wc -l`

In [ ]:
# how many resources were used per job? 
echo "Memory request vs usage:" 
condor_history -limit 3 -af RequestMemory MemoryUsage | sort | uniq -c
echo " " 
echo "Disk request vs usage:" 
condor_history -limit 3 -af RequestDisk MemoryDisk | sort | uniq -c

5. Place the full workload. 

In [ ]:
condor_submit jobs.sub -table 'jobtable.csv'